PydanticOutputParser를 래핑하고 이파서가 처리할 수 없는 형식의 출력이나 오류를 반환할 경우, 추가적인 llm 호출로 오류 수정

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List


# 프로젝트 이름을 입력합니다.
logging.langsmith("CH03-OutputFixParser")
load_dotenv()

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputFixParser


False

In [2]:
class Actor(BaseModel):
    name: str = Field(description="name of an actor")
    film_names: List[str] = Field(description="list of names of films they starred in")
    
actor_query = "Generate the filmography for a random actor."
parser = PydanticOutputParser(pydantic_object=Actor)


In [3]:
misformatted = "{'name':'Tom Hanks', 'film_names': ['Forrest Gump']}"

parser.parse(misformatted)

OutputParserException: Invalid json output: {'name':'Tom Hanks', 'film_names': ['Forrest Gump']}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [ ]:
from langchain_classic.output_parsers import OutputFixingParser
new_parser = OutputFixingParser.from_llm(parser=parser, llm=ChatOpenAI())

In [ ]:
actor = new_parser.parse(misformatted)

In [ ]:
actor

Actor(name='Tom Hanks', film_names=['Forrest Gump'])